# Andean Obsidian Geochemical Sourcing 
Welcome to the geochemical analysis notebook for XRF sourcing of obsidian from the Andes of South America.
This notebook provides the code for displaying geochemistry data together with obsidian source data from the region. It is intended to be reproducible and to provide a model for use with other data or analyses.

For long-term archival use, this notebook is paired with pinned dependency files in the repository root so it can be recreated in the future with a consistent Python environment.
Proceed through the notebook by clicking in the first cell and then press Shift-Enter to move through the notebook.
The Python can be modified and re-run, and maps and plots are interactive and can be saved.


In [ ]:
# SKIP THIS CURRENTLY
# 
# # CELL 0 — Environment bootstrap for local, Jupyter, and Voila usage.
# For archival reproducibility, prefer the pinned environment file in this folder.

from pathlib import Path
import os
import sys

if not os.environ.get("BINDER_LAUNCH_HOST"):
    req_path = Path.cwd() / "requirements.txt"
    if not req_path.exists():
        req_path = Path.cwd().parent / "requirements.txt"

    if req_path.exists():
        print(f"Using dependency file: {req_path}")
        print("Installing dependencies from requirements.txt...")
        %pip install -q -r {str(req_path)}
    else:
        print("No requirements.txt found. Create the environment with: conda env create -f environment.yml")
else:
    print("Binder-managed environment detected; dependencies come from environment.yml")


In [ ]:
# CELL 1 — Imports

import sys
import os
import io
import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    from plotly.graph_objects import FigureWidget
    _plotly_figure_widget_available = True
except Exception:
    FigureWidget = go.Figure
    _plotly_figure_widget_available = False

from plotly.colors import DEFAULT_PLOTLY_COLORS
from pathlib import Path
from ipywidgets import Button, Output, VBox, SelectMultiple, Layout, FileUpload, Text
import ipywidgets as widgets
from IPython.display import display, HTML

# Detect environment
def detect_environment():
    if os.environ.get("BINDER_LAUNCH_HOST"):
        return "binder"
    if os.environ.get("SERVER_SOFTWARE", "").startswith("voila"):
        return "voila"
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "vscode"
    return "jupyter"

ENV = detect_environment()
print(f"Environment : {ENV}")
print(f"Python      : {sys.version.split()[0]}")
print(f"FigureWidget: {'available' if _plotly_figure_widget_available else 'falling back to go.Figure'}")

# Scrollable output — only inject in environments that support it
if ENV in ("jupyter", "vscode", "binder"):
    display(HTML("""
        <style>
            .output_wrapper, .output { 
                max-height: 600px !important; 
                overflow-y: auto !important; 
            }
        </style>
    """))


## Loading tables from local CSV files
### Designed to load local CSV data by default from the `../data` folder:
*  **study_samples.csv** contains geochemistry of the study samples for the current investigation.
*  **South_Am_ObsSrcs_Chem_15-25Lat_only.csv** contains obsidian source geochemistry for 15-25° latitude south. 
*  **South_Am_ObsSrcs_Locs_15-25Lat_only.csv** contains obsidian source coordinates for 15-25° latitude south. 
*  **South_Am_ObsSrcs_Locs_all.csv** contains all known obsidian source coordinates for South America. 

## Data Loading

In [ ]:
# CELL 2 - DATA LOADING
# Upload CSV files for geochemical data and metadata

import io
from ipywidgets import FileUpload, Button, Output, VBox, HBox
import ipywidgets as widgets
from IPython.display import display, HTML

print("📤 DATA LOADING: Please upload your CSV files using the widgets below.\n")

def read_csv_with_encodings(content):
    """Read CSV content with multiple encoding attempts"""
    for encoding in ("utf-8", "latin1", "cp1252"):
        try:
            if isinstance(content, str):
                return pd.read_csv(io.StringIO(content), encoding=encoding)
            else:
                return pd.read_csv(io.BytesIO(content), encoding=encoding)
        except UnicodeDecodeError:
            continue
    if isinstance(content, str):
        return pd.read_csv(io.StringIO(content), encoding="latin1")
    else:
        return pd.read_csv(io.BytesIO(content), encoding="latin1")

# Global dataframes
srcs = pd.DataFrame()
srcs_locs = pd.DataFrame()
study = pd.DataFrame()

# --- 1. Sources Chemistry Upload ---
print("📁 1. Sources Chemistry (XRF Data)")
chem_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
chem_btn = widgets.Button(description="Load Sources Chemistry", button_style="success")
chem_out = widgets.Output()

def on_chem_click(b):
    global srcs
    if not chem_upload.value:
        with chem_out:
            print("⚠️ Please select a file first.")
        return
    try:
        uploaded = list(chem_upload.value.values())[0] if isinstance(chem_upload.value, dict) else chem_upload.value[0]
        content = uploaded.get("content")
        srcs = read_csv_with_encodings(content)
        with chem_out:
            print(f"✅ Loaded Sources Chemistry: {uploaded.get('name')} ({len(srcs)} rows, {len(srcs.columns)} cols)")
    except Exception as e:
        with chem_out:
            print(f"❌ Error: {e}")

chem_btn.on_click(on_chem_click)
display(HBox([chem_upload, chem_btn]), chem_out)

# --- 2. Sources Locations Upload ---
print("\n📁 2. Sources Locations (Coordinates)")
loc_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
loc_btn = widgets.Button(description="Load Sources Locations", button_style="success")
loc_out = widgets.Output()

def on_loc_click(b):
    global srcs_locs
    if not loc_upload.value:
        with loc_out:
            print("⚠️ Please select a file first.")
        return
    try:
        uploaded = list(loc_upload.value.values())[0] if isinstance(loc_upload.value, dict) else loc_upload.value[0]
        content = uploaded.get("content")
        raw_locs = read_csv_with_encodings(content)
        
        # Normalize location and schema fields
        rename_map = {
            "Chem_Group": "Group",
            "Latitude": "Lat",
            "Longitude": "Long",
            "Source": "Name"
        }
        raw_locs = raw_locs.rename(columns=rename_map)
        if "Name" not in raw_locs.columns and "Group" in raw_locs.columns:
            raw_locs["Name"] = raw_locs["Group"].astype("string")
        if "Group" in raw_locs.columns:
            raw_locs["Group"] = raw_locs["Group"].astype("string").str.strip()
        for col in ["Lat", "Long"]:
            if col in raw_locs.columns:
                raw_locs[col] = pd.to_numeric(raw_locs[col], errors="coerce")
        raw_locs = raw_locs.dropna(subset=[col for col in ["Lat", "Long"] if col in raw_locs.columns]).reset_index(drop=True)
        
        srcs_locs = raw_locs
        with loc_out:
            print(f"✅ Loaded Sources Locations: {uploaded.get('name')} ({len(srcs_locs)} rows)")
    except Exception as e:
        with loc_out:
            print(f"❌ Error: {e}")

loc_btn.on_click(on_loc_click)
display(HBox([loc_upload, loc_btn]), loc_out)

# --- 3. Study Samples Upload ---
print("\n📁 3. Study Samples (Unknowns)")
study_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
study_btn = widgets.Button(description="Load Study Samples", button_style="success")
study_out = widgets.Output()

def on_study_click(b):
    global study
    if not study_upload.value:
        with study_out:
            print("⚠️ Please select a file first.")
        return
    try:
        uploaded = list(study_upload.value.values())[0] if isinstance(study_upload.value, dict) else study_upload.value[0]
        content = uploaded.get("content")
        raw_study = read_csv_with_encodings(content)
        
        # Align study columns
        raw_study = raw_study.rename(columns={"Name": "Sample", "Application": "Group"})
        if "Group" not in raw_study.columns and "Application" in raw_study.columns:
            raw_study = raw_study.rename(columns={"Application": "Group"})
        if "Sample" not in raw_study.columns and "Name" in raw_study.columns:
            raw_study = raw_study.rename(columns={"Name": "Sample"})
            
        study = raw_study
        with study_out:
            print(f"✅ Loaded Study Samples: {uploaded.get('name')} ({len(study)} rows)")
    except Exception as e:
        with study_out:
            print(f"❌ Error: {e}")

study_btn.on_click(on_study_click)
display(HBox([study_upload, study_btn]), study_out)


In [ ]:
print (srcs.head())

### Cleaned-up Sample Data used appears below
Proprietary data headers are stripped out (i.e., Bruker headers removed), and schema enforced.

In [ ]:
# CELL 3 - DATA CLEANING
# Replace values below detection limits (often marked as <LOD or 0)
def clean_geochem_df(df):
    """Clean geochemistry dataframe headers and types."""
    if df is None or df.empty:
        return df

    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # Remove Bruker artifacts and spaces
    df.columns = df.columns.str.replace(r'(Ka1|La1|\s+)', '', regex=True)

    # String columns - assign column by column to avoid pandas multi-column assignment ValueError
    string_cols = ['Group', 'Sample', 'Name']
    for c in string_cols:
        if c in df.columns:
            df[c] = df[c].astype('string')

    # Numeric columns
    present_strings = [c for c in string_cols if c in df.columns]
    numeric_cols = [c for c in df.columns if c not in present_strings]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Drop all-NaN columns
    df = df.dropna(axis=1, how='all')
    return df

srcs = clean_geochem_df(srcs)
study = clean_geochem_df(study)

REQUIRED = {
    "srcs": ["Sample", "Group", "Rb", "Sr", "Zr"],
    "study": ["Sample", "Group", "Rb", "Sr", "Zr"]
}


def check_schema(df, required, name):
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

# Only enforce schema when the required columns are present in the data.
for df_name, df in [("Sources", srcs), ("Study", study)]:
    if df is not None and not df.empty:
        try:
            check_schema(df, REQUIRED["srcs" if df_name == "Sources" else "study"], df_name)
        except ValueError as exc:
            print(f"⚠️ {exc}")

print('Complete Data Types and Study Sample Table used in this visualization:')
display(study.dtypes)
display(study.head())

### Joined South American obsidian source chemistry with coordinates
The table below shows the source chemistry data joined to the source location coordinates by `Group`.

In [ ]:
# CELL 4 - JOIN SOURCE CHEMISTRY AND LOCATIONS
# Build a joined table from source chemistry and source locations

# Check if data is loaded
if srcs_locs is None or srcs_locs.empty:
    print("⚠️ Data not yet loaded")
    print("Please upload all required CSV files in the Data Loading cell above, then re-run this cell.")
    joined_srcs = pd.DataFrame()
else:
    location_columns = ["Group", "Lat", "Long", "Name"]
    missing_srcs_locs = [c for c in location_columns if c not in srcs_locs.columns]
    if missing_srcs_locs:
        print(f"⚠️ srcs_locs missing columns: {missing_srcs_locs}")
        print(f"Available columns: {srcs_locs.columns.tolist()}")
        joined_srcs = pd.DataFrame()
    else:
        srcs_locs_coords = srcs_locs[location_columns].drop_duplicates(subset=["Group"])
        srcs_locs_coords = srcs_locs_coords.rename(columns={"Lat": "Lat_loc", "Long": "Long_loc"})
        
        # Join source chemistry with locations
        joined_srcs = srcs.merge(
            srcs_locs_coords,
            on='Group',
            how='left'
        )
        
        print("✅ Joined source chemistry to source locations.")
        print(f"Rows: {len(joined_srcs)}")
        print(f"Columns: {joined_srcs.columns.tolist()}")

        if not joined_srcs.empty and "Lat" in joined_srcs.columns and "Long" in joined_srcs.columns:
            if joined_srcs[["Lat", "Long"]].isna().any(axis=None):
                print("⚠️ Some rows have missing coordinates after the join.")
            else:
                print("✅ All joined rows have coordinates.")
            display(joined_srcs.head(20))


In [ ]:
print(srcs_locs.head())

### Use Lasso tool to select obsidian sources from region of interest
Click Get Selection button below to continue

In [ ]:
# CELL 5 - SOURCE SELECTION WITH FIGUREWIDGET
# Monkey-patch FigureWidget handlers to fix UID KeyError and invalid mapbox._derived paths

# --- Patch function to skip deltas without 'uid' and filter invalid relayout paths ---
def _patched_delta_handler(fig_instance):
    """
    Monkey-patch FigureWidget handlers to:
    1. Skip trace deltas missing 'uid'
    2. Filter out invalid property paths like 'mapbox._derived' from relayout events
    3. Guard against Plotly versions where the relayout store is a plain dict instead of a widget
    This allows lasso selection to work without crashing.
    """
    # Patch 1: Handle trace deltas
    original_delta_handler = getattr(fig_instance, '_handler_js2py_traceDeltas', None)
    if original_delta_handler is not None:
        def safe_delta_handler(change):
            try:
                if isinstance(change.get('new'), list):
                    trace_deltas = change['new']
                    deltas_with_uid = [d for d in trace_deltas if isinstance(d, dict) and 'uid' in d]
                    if deltas_with_uid:
                        change_copy = dict(change)
                        change_copy['new'] = deltas_with_uid
                        try:
                            original_delta_handler(change_copy)
                        except (KeyError, IndexError):
                            pass
                else:
                    original_delta_handler(change)
            except Exception:
                pass

        fig_instance._handler_js2py_traceDeltas = safe_delta_handler
        trace_store = getattr(fig_instance, '_traceDeltas', None)
        if trace_store is not None and hasattr(trace_store, 'unobserve') and hasattr(trace_store, 'observe'):
            try:
                trace_store.unobserve(original_delta_handler, names='value')
            except Exception:
                pass
            try:
                trace_store.observe(safe_delta_handler, names='value')
            except Exception:
                pass

    # Patch 2: Handle relayout events to filter invalid property paths
    original_relayout_handler = getattr(fig_instance, '_handler_js2py_relayout', None)
    if original_relayout_handler is not None:
        def safe_relayout_handler(change):
            try:
                relayout_data = change.get('new', {})
                if isinstance(relayout_data, dict):
                    cleaned_relayout = {k: v for k, v in relayout_data.items() if '_derived' not in k}
                    if cleaned_relayout:
                        change_copy = dict(change)
                        change_copy['new'] = cleaned_relayout
                        original_relayout_handler(change_copy)
                else:
                    original_relayout_handler(change)
            except ValueError as e:
                if 'Invalid property path' in str(e):
                    pass
                else:
                    raise
            except Exception:
                pass

        fig_instance._handler_js2py_relayout = safe_relayout_handler
        relayout_store = getattr(fig_instance, '_js2py_relayout', None)
        if relayout_store is not None and hasattr(relayout_store, 'unobserve') and hasattr(relayout_store, 'observe'):
            try:
                relayout_store.unobserve(original_relayout_handler, names='value')
            except Exception:
                pass
            try:
                relayout_store.observe(safe_relayout_handler, names='value')
            except Exception:
                pass

# --- Data checks ---
if 'srcs_locs' not in globals():
    raise ValueError("Data not loaded: run the Data Loading cell first.")

srcs_locs = srcs_locs.copy()

if 'Lat' not in srcs_locs.columns or 'Long' not in srcs_locs.columns:
    raise ValueError("srcs_locs is missing Lat/Long columns.")

srcs_locs['Lat']  = pd.to_numeric(srcs_locs['Lat'],  errors='coerce')
srcs_locs['Long'] = pd.to_numeric(srcs_locs['Long'], errors='coerce')
srcs_locs = srcs_locs.dropna(subset=['Lat', 'Long'])

if srcs_locs.empty:
    raise ValueError("srcs_locs contains no valid coordinates after cleaning.")

names  = srcs_locs['Name'].astype(str).tolist()
groups = srcs_locs['Group'].astype(str).tolist()
lats   = srcs_locs['Lat'].astype(float).tolist()
lons   = srcs_locs['Long'].astype(float).tolist()

# DEBUG: Print first few coordinates
print(f"📍 First 3 coordinates:")
for i in range(min(3, len(names))):
    print(f"   {names[i]}: Lat={lats[i]}, Long={lons[i]}")
print(f"   Lat range: {min(lats):.2f} to {max(lats):.2f}")
print(f"   Long range: {min(lons):.2f} to {max(lons):.2f}")
print()

# --- Zoom helper ---
center_lat = np.mean(lats)
center_lon = np.mean(lons)
max_span   = max(max(lats) - min(lats), max(lons) - min(lons))
zoom = (5  if max_span > 10 else
        6  if max_span > 5  else
        7  if max_span > 2  else
        8  if max_span > 1  else
        9  if max_span > 0.5 else 11)

# --- Build trace with explicit UID ---
trace = go.Scattermapbox(
    lat=lats,
    lon=lons,
    mode='markers+text',
    text=names,
    textposition='top center',
    customdata=groups,
    marker=dict(size=10, color='steelblue'),
    selected=dict(marker=dict(size=14, color='red')),
    unselected=dict(marker=dict(opacity=0.4)),
    hovertemplate='<b>%{text}</b><br><b>Group:</b> %{customdata}<extra></extra>'
)

trace.uid = "scattermapbox_main"

# --- Create FigureWidget and apply monkey-patch ---
fig = FigureWidget(data=[trace])
_patched_delta_handler(fig)  # Apply patch to handle missing UIDs and invalid relayout paths

fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=center_lat, lon=center_lon),
        zoom=zoom
    ),
    dragmode='lasso',
    height=500,
    margin=dict(r=0, l=0, t=30, b=0),
    title="🗺️ Obsidian Source Locations — lasso to select, then click Get Selection",
    hovermode='closest'
)

display(fig)

# --- Selection widget ---
output = widgets.Output()
button = widgets.Button(
    description='Get Selection',
    button_style='primary',
    icon='check',
)

selected_names = []
selected_groups = []

def on_get_selection(b):
    global selected_names, selected_groups
    selected_names = []
    selected_groups = []
    
    output.clear_output()
    
    with output:
        try:
            sel = fig.data[0].selectedpoints if hasattr(fig, 'data') and len(fig.data) > 0 else None
            
            if not sel or len(sel) == 0:
                print("⚠️  No points selected. Draw a lasso on the map first.")
                return
            
            selected_names = [names[i] for i in sel]
            selected_groups = [groups[i] for i in sel]
            
            # Remove duplicates while preserving order
            seen = set()
            selected_groups = [g for g in selected_groups if not (g in seen or seen.add(g))]
            
            print(f"✅  {len(set(selected_groups))} source(s) selected:\n")
            for g in set(selected_groups):
                print(f"  • {g}")
        except Exception as e:
            print(f"❌ Error reading selection: {e}")

button.on_click(on_get_selection)

# Fallback checkboxes
unique_groups = sorted(list(set(groups)))
checkboxes = [widgets.Checkbox(value=False, description=g) for g in unique_groups]

def on_checkbox_change(change):
    global selected_names, selected_groups
    if change['new']:
        selected = [unique_groups[i] for i, cb in enumerate(checkboxes) if cb.value]
        if selected:
            selected_names = [n for n, g in zip(names, groups) if g in selected]
            selected_groups = list(set([g for g in groups if g in selected]))

for cb in checkboxes:
    cb.observe(on_checkbox_change, names='value')

checkbox_widget = widgets.VBox(checkboxes, layout=widgets.Layout(
    border='1px solid #ccc',
    padding='10px',
    height='150px',
    overflow_y='auto'
))

print("\n📌 Fallback: Check boxes below if lasso selection isn't working:")
display(checkbox_widget)
display(button, output)

## Select Obsidian Sources before proceeding
Use the map above this text to select obsidian sources in your region of interest, Click "Get Selection", then click in the cell below and continue running the notebook.

In [ ]:
# Display Selection results with error handling
if 'selected_names' in dir() and selected_names and len(selected_names) > 0:
    try:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #c8e6c9; border-radius: 5px; border-left: 4px solid #4CAF50;">
            <b>✅ Success!</b> Selected <b>{len(selected_names)}</b> source(s)
        </div>
        """))
        
        # Create summary table with error handling
        summary = []
        for name, group in zip(selected_names, selected_groups if 'selected_groups' in dir() else [None]*len(selected_names)):
            if group in srcs['Group'].values:
                count = len(srcs[srcs['Group'] == group])
                summary.append({"Source": name, "Group": group, "Count": count})
                print(f"  {name} | {group}: {count} samples")
            else:
                print(f"  ⚠️ Warning: {name} ({group}) not found in srcs data")
        
        if len(summary) > 0:
            summary_df = pd.DataFrame(summary).sort_values("Count", ascending=False)
            display(summary_df)
        else:
            display(HTML("""
            <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
                <b>❌ Error:</b> No selected sources found in data
            </div>
            """))
            
    except Exception as e:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
            <b>❌ Error processing selection:</b> {str(e)}
        </div>
        """))
        print(f"Full error: {e}")
else:
    display(HTML("""
    <div style="padding: 10px; background-color: #fff3e0; border-radius: 5px; border-left: 4px solid #FF9800;">
        <b>⚠️ No selections yet.</b> Click the button after selecting sources on the map.
    </div>
    """))

In [ ]:
# Apply selection with error handling
try:
    if 'selected_groups' not in dir() or not selected_groups:
        raise ValueError("No sources selected. Please select sources from the map first.")
    
    # Create subset using the Group values selected on the map
    srcs_subset = srcs[srcs['Group'].isin(selected_groups)].copy()

    # Prepare a normalized locations table from srcs_locs (handle varied column names)
    def _find_col(df, names):
        names_low = [n.lower() for n in names]
        for c in df.columns:
            if c.lower() in names_low:
                return c
        return None

    group_col = _find_col(srcs_locs, ['Group', 'group', 'Source', 'source']) or 'Group'
    lat_col = _find_col(srcs_locs, ['Lat', 'Latitude', 'lat', 'latitude', 'Lat_loc'])
    long_col = _find_col(srcs_locs, ['Long', 'Longitude', 'Long_loc', 'Lon', 'lon', 'longitude'])
    name_col = _find_col(srcs_locs, ['Name', 'name', 'Source', 'source'])

    if lat_col is None or long_col is None:
        raise KeyError(f"srcs_locs missing coordinate columns. Available columns: {list(srcs_locs.columns)}")

    loc_rename = {group_col: 'Group', lat_col: 'Lat', long_col: 'Long'}
    if name_col:
        loc_rename[name_col] = 'Name'
    srcs_locs_norm = srcs_locs.rename(columns=loc_rename)

    # Ensure Group exists in normalized locations
    if 'Group' not in srcs_locs_norm.columns:
        raise KeyError(f"Could not locate a 'Group' column in srcs_locs. Available: {list(srcs_locs.columns)}")

    srcs_locs_coords = srcs_locs_norm[['Group', 'Lat', 'Long'] + (['Name'] if 'Name' in srcs_locs_norm.columns else [])]
    srcs_locs_coords = srcs_locs_coords.drop_duplicates(subset=['Group']).reset_index(drop=True)

    # Now merge
    srcs_subset = srcs_subset.merge(
        srcs_locs_coords,
        on='Group',
        how='left'
    )
    
    if len(srcs_subset) == 0:
        raise ValueError(f"No data found for selected sources: {selected_groups}")
    
    if 'Lat' not in srcs_subset.columns or 'Long' not in srcs_subset.columns:
        print(f"⚠️ After merge, location columns missing. Available columns: {list(srcs_subset.columns)}")
    elif srcs_subset[['Lat', 'Long']].isna().any().any():
        print("⚠️ Some selected sources have no matching location data.")
    
    print(f"✅ Filtered to {len(srcs_subset)} samples from {len(selected_groups)} selected sources")
    
    # Count and split
    counts = srcs_subset['Group'].value_counts()
    onesample = srcs_subset[srcs_subset['Group'].map(counts) < 2]
    srcs_subset = srcs_subset[srcs_subset['Group'].map(counts) >= 2]
    
    print(f"✅ {len(srcs_subset)} samples for ellipses")
    print(f"✅ {len(onesample)} samples for points (< 2 per source)")
    
except ValueError as e:
    print(f"❌ Error: {e}")
    srcs_subset = pd.DataFrame()  # Empty dataframe
except KeyError as e:
    print(f"❌ Key error: {e}")
    srcs_subset = pd.DataFrame()
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {e}")
    srcs_subset = pd.DataFrame()

In [ ]:
# This cell contains code for creating 1 s.d. confidence ellipses on biplots

def confidence_ellipse(x, y, n_std=1.96, size=100):   # Ellipses in Plotly
    """
    Get the covariance confidence ellipse of *x* and *y*.
    from https://gist.github.com/dpfoose/38ca2f5aee2aea175ecc6e599ca6e973

    Parameters
    ----------
    x, y : array-like, shape (n, )
        Input data.
    n_std : float
        The number of standard deviations to determine the ellipse's radiuses.
    size : int
        Number of points defining the ellipse
    Returns
    -------
    String containing an SVG path for the ellipse

    References (H/T)
    ----------------
    https://matplotlib.org/3.1.1/gallery/statistics/confidence_ellipse.html
    https://community.plotly.com/t/arc-shape-with-path/7205/5
    """
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    pearson = cov[0, 1]/np.sqrt(cov[0, 0] * cov[1, 1])
    # Using a special case to obtain the eigenvalues of this
    # two-dimensionl dataset.
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    theta = np.linspace(0, 2 * np.pi, size)
    ellipse_coords = np.column_stack([ell_radius_x * np.cos(theta), ell_radius_y * np.sin(theta)])

    # Calculating the stdandard deviation of x from
    # the squareroot of the variance and multiplying
    # with the given number of standard deviations.
    x_scale = np.sqrt(cov[0, 0]) * n_std
    x_mean = np.mean(x)

    # calculating the stdandard deviation of y ...
    y_scale = np.sqrt(cov[1, 1]) * n_std
    y_mean = np.mean(y)

    translation_matrix = np.tile([x_mean, y_mean], (ellipse_coords.shape[0], 1))
    rotation_matrix = np.array([[np.cos(np.pi / 4), np.sin(np.pi / 4)],
                                [-np.sin(np.pi / 4), np.cos(np.pi / 4)]])
    scale_matrix = np.array([[x_scale, 0],
                            [0, y_scale]])
    ellipse_coords = ellipse_coords.dot(rotation_matrix).dot(scale_matrix) + translation_matrix

    path = f'M {ellipse_coords[0, 0]}, {ellipse_coords[0, 1]}'
    for k in range(1, len(ellipse_coords)):
        path += f'L{ellipse_coords[k, 0]}, {ellipse_coords[k, 1]}'
    path += ' Z'
    return path



In [ ]:
# Assign a color to each Source for consistency
name_to_color = {}

unique_groups = srcs['Group'].dropna().unique()  # drop NA just in case
colors = DEFAULT_PLOTLY_COLORS

# Cycle colors if more groups than colors
name_to_color = {name: colors[i % len(colors)] for i, name in enumerate(unique_groups)}

# 2️⃣ If you want a simple mapping to original name (optional)
unique_name_to_name = {name: name for name in unique_groups}

In [ ]:
# BIPLOT

# Remove duplicate column names.
def _remove_duplicate_columns(df):
    if not isinstance(df, pd.DataFrame):
        return pd.DataFrame()

    result = df.copy()
    result.columns = result.columns.astype(str).str.strip()
    return result.loc[:, ~result.columns.duplicated(keep="first")]


srcs = _remove_duplicate_columns(srcs)
study = _remove_duplicate_columns(study)

if "srcs_subset" in globals():
    srcs_subset = _remove_duplicate_columns(srcs_subset)

srcs_subset_valid = (
    srcs_subset.copy()
    if (
        "srcs_subset" in globals()
        and isinstance(srcs_subset, pd.DataFrame)
        and not srcs_subset.empty
    )
    else pd.DataFrame()
)

GROUP_COL = "Group"
plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]

available_elements = [
    col for col in plot_elements
    if col in srcs.columns and col in study.columns
]

if not available_elements:
    raise ValueError(
        "No common geochemical columns are available in srcs and study."
    )

x_default = "Sr" if "Sr" in available_elements else available_elements[0]
y_default = "Rb" if "Rb" in available_elements else available_elements[0]

x_menu = widgets.Dropdown(
    options=available_elements,
    value=x_default,
    description="X axis:",
    style={"description_width": "initial"}
)

y_menu = widgets.Dropdown(
    options=available_elements,
    value=y_default,
    description="Y axis:",
    style={"description_width": "initial"}
)

display(widgets.HBox([x_menu, y_menu]))


def _ellipse_points(x, y, n_points=120, n_std=1.0):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return None

    covariance = np.cov(x, y)

    if not np.isfinite(covariance).all():
        return None

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    eigenvalues = np.maximum(eigenvalues, 0)

    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    theta = np.linspace(0, 2 * np.pi, n_points)
    unit_circle = np.column_stack([
        np.cos(theta),
        np.sin(theta)
    ])

    ellipse = (
        unit_circle
        @ np.diag(n_std * np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    ellipse += [x.mean(), y.mean()]

    return ellipse[:, 0], ellipse[:, 1]


def build_biplot(x_col, y_col):
    fig = go.Figure()
    study_trace_indices = []

    source_data = (
        srcs_subset_valid.copy()
        if (
            isinstance(srcs_subset_valid, pd.DataFrame)
            and not srcs_subset_valid.empty
            and GROUP_COL in srcs_subset_valid.columns
        )
        else pd.DataFrame()
    )

    # Add source ellipses.
    if (
        not source_data.empty
        and x_col in source_data.columns
        and y_col in source_data.columns
    ):
        source_groups = (
            source_data[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for source_group in source_groups:
            group_data = source_data[
                source_data[GROUP_COL].astype(str) == source_group
            ]

            ellipse = _ellipse_points(
                group_data[x_col],
                group_data[y_col],
                n_points=120,
                n_std=1.0
            )

            if ellipse is None:
                continue

            ellipse_x, ellipse_y = ellipse
            color = name_to_color.get(source_group, "gray")

            fig.add_trace(go.Scatter(
                x=ellipse_x,
                y=ellipse_y,
                mode="lines",
                fill="toself",
                fillcolor=color,
                line=dict(color=color, width=2),
                opacity=0.35,
                name=f"{source_group} Source Ellipse",
                hoverinfo="skip"
            ))

            numeric_x = pd.to_numeric(
                group_data[x_col],
                errors="coerce"
            )

            numeric_y = pd.to_numeric(
                group_data[y_col],
                errors="coerce"
            )

            fig.add_trace(go.Scatter(
                x=[numeric_x.mean()],
                y=[numeric_y.mean()],
                mode="text",
                text=[f"{source_group} Source"],
                textposition="middle center",
                textfont=dict(size=11, color=color),
                showlegend=False,
                hoverinfo="skip"
            ))

    # Add source samples with fewer than two observations.
    if (
        "onesample" in globals()
        and isinstance(onesample, pd.DataFrame)
        and not onesample.empty
        and x_col in onesample.columns
        and y_col in onesample.columns
    ):
        valid = (
            pd.to_numeric(onesample[x_col], errors="coerce").notna()
            & pd.to_numeric(onesample[y_col], errors="coerce").notna()
        )

        source_sample_text = (
            onesample.loc[valid, "Sample"]
            if "Sample" in onesample.columns
            else None
        )

        fig.add_trace(go.Scatter(
            x=pd.to_numeric(
                onesample.loc[valid, x_col],
                errors="coerce"
            ),
            y=pd.to_numeric(
                onesample.loc[valid, y_col],
                errors="coerce"
            ),
            name="Source Sample",
            mode="markers",
            marker=dict(symbol="x", size=8, color="black"),
            text=source_sample_text,
            hovertemplate="Source: %{text}<br><extra></extra>",
            showlegend=True
        ))

    # Add study samples grouped by Group.
    if (
        isinstance(study, pd.DataFrame)
        and not study.empty
        and GROUP_COL in study.columns
        and x_col in study.columns
        and y_col in study.columns
    ):
        study_groups = (
            study[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for study_group in study_groups:
            group_data = study[
                study[GROUP_COL].astype(str) == study_group
            ].copy()

            x_values = pd.to_numeric(
                group_data[x_col],
                errors="coerce"
            )

            y_values = pd.to_numeric(
                group_data[y_col],
                errors="coerce"
            )

            valid = x_values.notna() & y_values.notna()

            if not valid.any():
                continue

            color = name_to_color.get(study_group, "gray")

            sample_text = (
                group_data.loc[valid, "Sample"]
                if "Sample" in group_data.columns
                else None
            )

            study_trace_indices.append(len(fig.data))

            fig.add_trace(go.Scatter(
                x=x_values.loc[valid],
                y=y_values.loc[valid],
                name=f"Study: {study_group}",
                mode="markers",
                text=sample_text,
                hovertemplate="Sample: %{text}<br><extra></extra>",
                marker=dict(
                    size=8,
                    symbol="circle",
                    color=color
                ),
                showlegend=True
            ))

    fig.update_layout(
        title=dict(
            text="Connecting Study Samples with known obsidian sources",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title=x_col,
        yaxis_title=y_col,
        height=600,
        margin=dict(
            l=0,
            r=240,
            t=120,
            b=0
        ),
        legend=dict(
            x=1.02,
            y=0.98,
            xanchor="left",
            yanchor="top"
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="down",
                x=1.02,
                y=1.08,
                xanchor="left",
                yanchor="bottom",
                buttons=[
                    dict(
                        label="Labels OFF",
                        method="restyle",
                        args=[
                            {"mode": "markers"},
                            study_trace_indices
                        ]
                    ),
                    dict(
                        label="Labels ON",
                        method="restyle",
                        args=[
                            {
                                "mode": "markers+text",
                                "textposition": "top center"
                            },
                            study_trace_indices
                        ]
                    )
                ]
            )
        ]
    )

    return fig


biplot_output = widgets.Output()


def update_biplot(change=None):
    with biplot_output:
        biplot_output.clear_output(wait=True)
        display(build_biplot(x_menu.value, y_menu.value))


x_menu.observe(update_biplot, names="value")
y_menu.observe(update_biplot, names="value")

display(biplot_output)
update_biplot()

### Biplot generated for visual source identification
Investigate biplot shown above. You may change the element variables at start of previous cell and re-run cell in order to view source ellipses and study samples using different elements on X and Y axes.
Proceed to Ternary diagram below to view three element variables at once.

In [ ]:
# TERNARY PLOT CODE

GROUP_COL = "Group"


def normalize_composition(df, cols):
    values = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    totals = np.sum(values, axis=1, keepdims=True)

    valid = np.isfinite(totals[:, 0]) & (totals[:, 0] > 0)
    result = np.full_like(values, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        result[valid] = values[valid] / totals[valid]

    return result


def confidence_ellipse_points(x, y, n_std=1.96, size=100):
    if len(x) < 2:
        return None

    covariance = np.cov(x, y)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    eigenvalues = np.maximum(eigenvalues, 0)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    theta = np.linspace(0, 2 * np.pi, size)
    unit_circle = np.column_stack([np.cos(theta), np.sin(theta)])

    ellipse = (
        unit_circle
        @ np.diag(n_std * np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    ellipse += [np.mean(x), np.mean(y)]
    return ellipse


def build_ternary_plot(a_col, b_col, c_col):
    cols = [a_col, b_col, c_col]
    fig = go.Figure()
    study_trace_indices = []

    source_data = srcs_subset_valid.copy()
    source_fraction = normalize_composition(source_data, cols)
    study_fraction = normalize_composition(study, cols)

    # Source ellipses
    if not source_data.empty and GROUP_COL in source_data.columns:
        for source_group in source_data[GROUP_COL].dropna().unique():
            group_mask = (
                source_data[GROUP_COL].astype(str)
                == str(source_group)
            ).to_numpy()

            data = source_fraction[group_mask]
            valid = np.isfinite(data).all(axis=1)
            data = data[valid]

            if len(data) < 2:
                continue

            ellipse = confidence_ellipse_points(
                data[:, 0],
                data[:, 1],
                n_std=1.0
            )

            if ellipse is None:
                continue

            a_values = ellipse[:, 0]
            b_values = ellipse[:, 1]
            c_values = 1 - a_values - b_values

            color = name_to_color.get(source_group, "gray")

            fig.add_trace(go.Scatterternary(
                a=a_values,
                b=b_values,
                c=c_values,
                mode="lines",
                line=dict(color=color),
                fill="toself",
                opacity=0.35,
                name=f"{source_group} Source"
            ))

    # Study samples
    if not study.empty and GROUP_COL in study.columns:
        for study_group in study[GROUP_COL].dropna().unique():
            group_mask = (
                study[GROUP_COL].astype(str)
                == str(study_group)
            ).to_numpy()

            data = study_fraction[group_mask]
            valid = np.isfinite(data).all(axis=1)
            data = data[valid]

            if len(data) == 0:
                continue

            color = name_to_color.get(study_group, "gray")
            trace_index = len(fig.data)

            fig.add_trace(go.Scatterternary(
                a=data[:, 0],
                b=data[:, 1],
                c=data[:, 2],
                mode="markers",
                name=f"Study: {study_group}",
                marker=dict(size=8, color=color),
                showlegend=True
            ))

            study_trace_indices.append(trace_index)

    fig.update_layout(
        ternary=dict(
            sum=1,
            aaxis=dict(title=a_col),
            baxis=dict(title=b_col),
            caxis=dict(title=c_col)
        ),
        title=dict(
            text=f"Ternary Plot: {a_col}, {b_col}, {c_col}",
            x=0.5,
            xanchor="center"
        ),
        legend=dict(
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top"
        ),
        height=550,
        updatemenus=[
            dict(
                type="buttons",
                direction="down",
                x=1.02,
                y=1.16,
                xanchor="left",
                yanchor="bottom",
                buttons=[
                    dict(
                        label="Labels OFF",
                        method="restyle",
                        args=[{"mode": "markers"}, study_trace_indices]
                    ),
                    dict(
                        label="Labels ON",
                        method="restyle",
                        args=[
                            {
                                "mode": "markers+text",
                                "textposition": "top center"
                            },
                            study_trace_indices
                        ]
                    )
                ]
            )
        ]
    )

    return fig


ternary_elements = [
    column for column in plot_elements
    if column in srcs.columns and column in study.columns
]

ternary_a_menu = widgets.Dropdown(
    options=ternary_elements,
    value="Rb",
    description="A axis:"
)

ternary_b_menu = widgets.Dropdown(
    options=ternary_elements,
    value="Sr",
    description="B axis:"
)

ternary_c_menu = widgets.Dropdown(
    options=ternary_elements,
    value="Zr",
    description="C axis:"
)

display(widgets.HBox([
    ternary_a_menu,
    ternary_b_menu,
    ternary_c_menu
]))

ternary_output = widgets.Output()


def update_ternary(change=None):
    with ternary_output:
        ternary_output.clear_output(wait=True)
        display(build_ternary_plot(
            ternary_a_menu.value,
            ternary_b_menu.value,
            ternary_c_menu.value
        ))


ternary_a_menu.observe(update_ternary, names="value")
ternary_b_menu.observe(update_ternary, names="value")
ternary_c_menu.observe(update_ternary, names="value")

display(ternary_output)
update_ternary()

### Ternary Plot
Examine relationship between samples and the ellipses representing known sources. Zoom into the Ternary diagram and change the variables shown in the previous cell if you wish to view other elements in the diagram.

### The analysis notebook has concluded.
In your data exploration mouse over the points in the biplot and ternary plot to view the unique ID numbers of those samples and note their relationship to ellipses for known obsidian sources.